Step 1: Install all the required package

In [4]:
!pip install -q accelerate peft bitsandbytes transformers trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.1 MB/s eta 0:00:00


Step 2: Import all the required libraries

In [5]:
import os
import torch
from datasets import load_dataset
from transformers import(
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

# **In case of LLama 2, the following prompt template is used for the chat models**

system prompt (optional) to guide the model,
user prompt (required) to give the instruction,
model answer (required)

s> [INST] <<SYS>>
System Prompt
<</SYS>>

User prompt [/INST] Model answer /s>

#We will reformat our instruction dataset to follow Llama 2 template.

Orignal Dataset: https://huggingface.co/datasets/timdettmers/openassistant-guanaco
add Codeadd Markdown
Reformat Dataset following the Llama 2 template with 1k sample: https://huggingface.co/datasets/mlabonne/guanaco-llama2-1k
add Codeadd Markdown
Complete Reformat Dataset following the Llama 2 template: https://huggingface.co/datasets/mlabonne/guanaco-llama2
add Codeadd Markdown
To know how this dataset was created, you can check this notebook.

https://colab.research.google.com/drive/1Ad7a9zMmkxuXTOh1Z7-rNSICA4dybpM2?usp=sharing

Note: You don’t need to follow a specific prompt template if you’re using the base Llama 2 model instead of the chat version.
add Codeadd Markdown
#How to fine tune Llama 2

add Codeadd Markdown
Free Google Colab offers a 15GB Graphics Card (Limited Resources --> Barely enough to store Llama 2–7b’s weights)
add Codeadd Markdown
We also need to consider the overhead due to optimizer states, gradients, and forward activations
add Codeadd Markdown
Full fine-tuning is not possible here: we need parameter-efficient fine-tuning (PEFT) techniques like LoRA or QLoRA.
add Codeadd Markdown
To drastically reduce the VRAM usage, we must fine-tune the model in 4-bit precision, which is why we’ll use QLoRA here.
add Codeadd Markdown
#Step 3

add Codeadd Markdown
Load a llama-2-7b-chat-hf model (chat model)
Train it on the mlabonne/guanaco-llama2-1k (1,000 samples), which will produce our fine-tuned model Llama-2-7b-chat-finetune
add Codeadd Markdown
QLoRA will use a rank of 64 with a scaling parameter of 16. We’ll load the Llama 2 model directly in 4-bit precision using the NF4 type and train it for one epoch

add Codeadd Markdown

In [8]:
# The mode that you want to train from the Hugging Face hub
model_name = 'NousResearch/Llama-2-7b-chat-hf'

# The instruction dataset to use
dataset_name = 'mlabonne/guanaco-llama2-1k'

# Fine-tuned model name
new_model = 'Llama-2-7b-chat-finetune'

QLoRA parameters

In [10]:
# lora attention dimension
lora_r = 64

# alpha parameter for LoRA scaling
lora_alpha = 16

# dropout probability for LoRA layers
lora_dropout = 0.1

BitsandBytes parameters

In [11]:
# activate 4-bit percision base model loading
use_4bit = True

# compute dtype for 4-bit base models
bnb_4bit_compute_dtype = 'float16'

# quantization type (fp4 or nf4)
bnb_4bit_quant_type = 'nf4'

# activate nested quantized for 4-bit based models (double quanitzed)
use_nested_quant = False

TrainingArguments parameters

In [12]:
# output directory where the model predicitons and checkpoints will be stored

output_dir = './results'

# number of training epochs
num_train_epochs = 1

# enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = False

# batch size per GPU for training
per_device_train_batch_size = 4

# batch size per GPU for evaluation
per_device_eval_batch_size = 4

# number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# enable gradient checkpointing
gradient_checkpointing = True

# maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# weights_decay to apply to all layers expect bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = 'paged_adamw_32bit'

# Learning rate schedule
lr_schedular_type = 'cosine'

# number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches
# saves memory and speeds up training considerably
